In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler

from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor


In [2]:

# Download all kinases from API

url = "https://kinepik.org/api/0/kinases/all"

response = requests.get(url)

# Convert API response to DataFrame
kinase_df = pd.DataFrame(response.json())

# Keep only UniProt ID and Gene Symbol
kinase_df = pd.DataFrame({
    "UniprotID": kinase_df["UniprotID"],
    "GeneName": kinase_df["GeneInfo"].apply(lambda x: x["MappedGene"])
})

# Display table
kinase_df.head()

,UniprotID,GeneName
0,P06239,LCK
1,P12931,SRC
2,P06241,FYN
3,P00519,ABL1
4,P24941,CDK2


In [3]:
print("Total Kinases:", len(kinase_df))

Total Kinases: 504


In [4]:
# get phosphosites for all kinases

def get_phosphosites(kinase_id):

    url = ( "https://kinepik.org/api/0/kinases/specific?"
        f"kinase_ids={kinase_id}&phosphosites=sites"
    )

    response = requests.get(url)
    data = response.json()

    if len(data) ==0:
        return []

    return data[0]["PhosphositesOnKinase"]

In [5]:

# Collect phosphosites for all 504 kinases

all_phosphosites = []

for _, row in kinase_df.iterrows():

    kinase_id = row["UniprotID"]
    gene = row["GeneName"]

    sites = get_phosphosites(kinase_id)

    all_phosphosites.append({
        "UniprotID": kinase_id,
        "GeneName": gene,
        "TotalPhosphosites": len(sites),
        "Phosphosites": sites
    })

phosphosite_df = pd.DataFrame(all_phosphosites)

phosphosite_df.head(10)

,UniprotID,GeneName,TotalPhosphosites,Phosphosites
0,P06239,LCK,6,"[LCK(Y394), LCK(Y505), LCK(Y192), LCK(S42), LC..."
1,P12931,SRC,17,"[SRC(Y216), SRC(Y338), SRC(Y419), SRC(Y530), S..."
2,P06241,FYN,13,"[FYN(Y39), FYN(Y420), FYN(Y28), FYN(Y30), FYN(..."
3,P00519,ABL1,37,"[ABL1(S446), ABL1(S465), ABL1(Y393), ABL1(Y226..."
4,P24941,CDK2,9,"[CDK2(T160), CDK2(Y168), CDK2(S46), CDK2(T165)..."
5,O14757,CHEK1,16,"[CHEK1(S301), CHEK1(S286), CHEK1(S345), CHEK1(..."
6,Q96GD4,AURKB,8,"[AURKB(T232), AURKB(T16), AURKB(S7), AURKB(S33..."
7,P06493,CDK1,12,"[CDK1(S39), CDK1(T161), CDK1(T222), CDK1(Y15),..."
8,O15530,PDPK1,26,"[PDPK1(T513), PDPK1(S241), PDPK1(S393), PDPK1(..."
9,P07949,RET,16,"[RET(Y809), RET(Y1090), RET(Y826), RET(Y1029),..."


In [6]:
# Save phosphosite counts

phosphosite_df.to_csv(
    "phosphosite_counts.csv",
    index=False
)

print("Saved phosphosite_counts.csv")

Saved phosphosite_counts.csv


In [6]:
# get fc values for one phosphosite

def get_fc(phosphosite):

    url = ( "https://kinepik.org/api/0/perturbation/fc?"
        f"type=target_phosphosite&id={phosphosite}"
        "&cell_line=MCF7&confidence=1"
    )

    response = requests.get(url)

    return response.json()

In [ ]:
# Download FC data for all phosphosites

fc_rows = []

for _, row in phosphosite_df.iterrows():

    kinase_id = row["UniprotID"]
    gene = row["GeneName"]

    for site in row["Phosphosites"]:

        try:
            fc_data = get_fc(site)

            for record in fc_data:

                info = record[site]

                fc_rows.append({
                    "UniprotID": kinase_id,
                    "GeneName": gene,
                    "Phosphosite": site,
                    "Perturbation": info["Perturbation"],
                    "CellLine": info["CellLine"],
                    "FC": float(info["FC"])   # <-- FIXED
                })

        except:
            continue

fc_df = pd.DataFrame(fc_rows)

In [8]:
print("Number of FC rows:", len(fc_rows))

Number of FC rows: 46238


In [9]:
print(fc_df.shape)

fc_df.head()

fc_df.columns

fc_df.info()

(46238, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46238 entries, 0 to 46237
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   UniprotID     46238 non-null  object 
 1   GeneName      46238 non-null  object 
 2   Phosphosite   46238 non-null  object 
 3   Perturbation  46238 non-null  object 
 4   CellLine      46238 non-null  object 
 5   FC            46238 non-null  float64
dtypes: float64(1), object(5)
memory usage: 2.1+ MB


In [10]:
ksea_rows = []

In [11]:
# Download KSEA for every kinase

for _, row in kinase_df.iterrows():

    kinase_id = row["UniprotID"]
    gene = row["GeneName"]

    # only perturbations that exist for this kinase
    kinase_fc = fc_df[fc_df["UniprotID"] == kinase_id]

    perturbations = kinase_fc["Perturbation"].unique()

    print(f"{gene}: {len(perturbations)} perturbations")

    for pert in perturbations:

        try:

            url = (
                f"https://kinepik.org/api/0/perturbation/KSEA?"
                f"kinase_ids={kinase_id}"
                f"&perturbations={pert}"
                f"&cell_line=MCF7"
            )

            response = requests.get(url)
            data = response.json()

            z_score = data[0][kinase_id][pert]["z_score"]

            ksea_rows.append({
                "UniprotID": kinase_id,
                "GeneName": gene,
                "Perturbation": pert,
                "CellLine": "MCF7",
                "KSEA_z_score": z_score
            })

        except:
            continue

LCK: 0 perturbations
SRC: 61 perturbations
FYN: 0 perturbations
ABL1: 61 perturbations
CDK2: 61 perturbations
CHEK1: 61 perturbations
AURKB: 0 perturbations
CDK1: 61 perturbations
PDPK1: 61 perturbations
RET: 61 perturbations
MAP3K7: 61 perturbations
PRKCQ: 61 perturbations
IKBKB: 61 perturbations
JAK2: 61 perturbations
CHEK2: 0 perturbations
ATM: 61 perturbations
TTK: 61 perturbations
PLK1: 61 perturbations
CAMK2A: 0 perturbations
MAPK3: 61 perturbations
MAPK1: 61 perturbations
EGFR: 61 perturbations
PRKCA: 61 perturbations
CSNK2A1: 0 perturbations
GSK3A: 61 perturbations
GSK3B: 61 perturbations
INSR: 0 perturbations
UHMK1: 0 perturbations
AKT2: 0 perturbations
PAK1: 61 perturbations
MAPK14: 0 perturbations
CDK7: 61 perturbations
KSR1: 61 perturbations
PAK3: 0 perturbations
PRKDC: 61 perturbations
ERBB2: 61 perturbations
NTRK1: 0 perturbations
MAPKAPK5: 61 perturbations
VRK1: 0 perturbations
DYRK2: 61 perturbations
HIPK2: 61 perturbations
AURKA: 0 perturbations
DAPK1: 0 perturbations


In [13]:
print(type(ksea_rows))
print(len(ksea_rows))

<class 'list'>
13786


In [14]:
ksea_df = pd.DataFrame(ksea_rows)

In [ ]:
print(ksea_df.shape)

ksea_df.head()

ksea_df.columns

ksea_df.info()

(13786, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13786 entries, 0 to 13785
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   UniprotID     13786 non-null  object 
 1   GeneName      13786 non-null  object 
 2   Perturbation  13786 non-null  object 
 3   CellLine      13786 non-null  object 
 4   KSEA_z_score  10431 non-null  float64
dtypes: float64(1), object(4)
memory usage: 538.6+ KB


In [15]:
#Merge FC+KSEA

merged_df = fc_df.merge(
    ksea_df,
    on=["UniprotID", "GeneName", "Perturbation", "CellLine"],
    how="inner" 
)

In [16]:
print(merged_df.shape)

merged_df.head()

(46238, 7)


,UniprotID,GeneName,Phosphosite,Perturbation,CellLine,FC,KSEA_z_score
0,P12931,SRC,SRC(S75),AZD6482,MCF7,-0.261965,1.482936
1,P12931,SRC,SRC(S75),CAL101,MCF7,-0.455884,-0.187255
2,P12931,SRC,SRC(S75),GDC0941,MCF7,2.216590,1.303407
3,P12931,SRC,SRC(S75),HS173,MCF7,-0.266605,0.125534
4,P12931,SRC,SRC(S75),PIK294,MCF7,0.520571,0.845103


In [17]:
merged_df

,UniprotID,GeneName,Phosphosite,Perturbation,CellLine,FC,KSEA_z_score
0,P12931,SRC,SRC(S75),AZD6482,MCF7,-0.261965,1.482936
1,P12931,SRC,SRC(S75),CAL101,MCF7,-0.455884,-0.187255
2,P12931,SRC,SRC(S75),GDC0941,MCF7,2.216590,1.303407
3,P12931,SRC,SRC(S75),HS173,MCF7,-0.266605,0.125534
4,P12931,SRC,SRC(S75),PIK294,MCF7,0.520571,0.845103
...,...,...,...,...,...,...,...
46233,Q96L96,ALPK3,ALPK3(S430),Vemurafenib,MCF7,-0.159849,NaN
46234,Q96L96,ALPK3,ALPK3(S430),CX4945,MCF7,-1.434684,NaN
46235,Q96L96,ALPK3,ALPK3(S430),GO6983,MCF7,-1.434684,NaN
46236,Q96L96,ALPK3,ALPK3(S430),KN93,MCF7,-1.434684,NaN


In [18]:
# Remove rows with missing KSEA

usable_df = merged_df.dropna(subset=["KSEA_z_score"]).copy()

print(usable_df.shape)

(35990, 7)


In [19]:
#Count phosphosites per kinase

phosphosite_count = (
    usable_df
    .groupby("UniprotID")["Phosphosite"]
    .nunique()
    .reset_index()

)

phosphosite_count.columns = ["UniprotID", "Num_Phosphosites"]

phosphosite_count

,UniprotID,Num_Phosphosites
0,O00418,5
1,O00506,1
2,O14578,6
3,O14733,2
4,O14757,2
...,...,...
166,Q9Y463,1
167,Q9Y4K4,2
168,Q9Y572,2
169,Q9Y5S2,1


In [20]:
print(phosphosite_count.shape)

phosphosite_count.info()

(171, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171 entries, 0 to 170
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   UniprotID         171 non-null    object
 1   Num_Phosphosites  171 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 2.8+ KB


In [21]:
usable_df

,UniprotID,GeneName,Phosphosite,Perturbation,CellLine,FC,KSEA_z_score
0,P12931,SRC,SRC(S75),AZD6482,MCF7,-0.261965,1.482936
1,P12931,SRC,SRC(S75),CAL101,MCF7,-0.455884,-0.187255
2,P12931,SRC,SRC(S75),GDC0941,MCF7,2.216590,1.303407
3,P12931,SRC,SRC(S75),HS173,MCF7,-0.266605,0.125534
4,P12931,SRC,SRC(S75),PIK294,MCF7,0.520571,0.845103
...,...,...,...,...,...,...,...
42634,Q13263,TRIM28,TRIM28(T418),Vemurafenib,MCF7,-0.145543,-0.248085
42635,Q13263,TRIM28,TRIM28(T418),CX4945,MCF7,1.737539,-0.198538
42636,Q13263,TRIM28,TRIM28(T418),GO6983,MCF7,1.984330,0.248577
42637,Q13263,TRIM28,TRIM28(T418),KN93,MCF7,1.691755,0.466063


In [22]:
#keep kinases with at least 3 phosphosites

usable_kinases = phosphosite_count[
    phosphosite_count["Num_Phosphosites"] >= 3
]

print(usable_kinases.shape)

usable_kinases

(85, 2)


,UniprotID,Num_Phosphosites
0,O00418,5
2,O14578,6
7,O15075,7
11,O43318,4
12,O43353,4
...,...,...
155,Q9NYV4,22
161,Q9UKE5,6
163,Q9Y2K2,8
164,Q9Y2U5,8


In [23]:
usable_kinases.info()

<class 'pandas.core.frame.DataFrame'>
Index: 85 entries, 0 to 165
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   UniprotID         85 non-null     object
 1   Num_Phosphosites  85 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 2.0+ KB


In [24]:
# store one feature matrix for each kinase
feature_matrices = {}

In [25]:
#Build one matrix per kinase

for kinase_id in usable_kinases["UniprotID"]:
    
    #Get data for one kinase
    kinase_data = usable_df[
        usable_df["UniprotID"] ==kinase_id 
    ].copy()

    # Convert long format to wide format
    kinase_matrix = kinase_data.pivot_table(
        index="Perturbation",
        columns="Phosphosite",
        values="FC",
        aggfunc="first" 
    )

    #Get one KSEA value for each perturbation
    ksea = (
        kinase_data[
            ["Perturbation","KSEA_z_score"] 
        ]
        .drop_duplicates()
        .set_index("Perturbation") 
    )

    # Join FC features with KSEA target
    kinase_matrix = kinase_matrix.join(ksea)

    #Store using Uniprot ID
    feature_matrices[kinase_id] = kinase_matrix

print(f"Created {len(feature_matrices)} feature matrices.")

Created 85 feature matrices.


In [26]:
print(len(feature_matrices))

85


In [27]:
# Let's inspect one matrix

first_kinase = list(feature_matrices.keys())[0]

print(first_kinase)

feature_matrices[first_kinase]

O00418


,EEF2K(S18),EEF2K(S66),EEF2K(S72),EEF2K(S74),EEF2K(Y69),KSEA_z_score
Perturbation,,,,,,
AC220,0.664402,-1.563777,-1.686784,-1.405499,-0.623428,-1.904592
AT13148,0.045253,-0.191064,-0.219305,-0.263898,-0.317240,-1.795995
AZ20,0.250962,0.096789,-8.797069,-4.829727,-7.902993,3.515267
AZD1480,-0.319168,-1.563777,-0.024642,-0.047332,-0.123976,-0.641596
AZD3759,-0.618604,-2.529060,-0.435913,-0.603194,-1.625165,-0.819098
...,...,...,...,...,...,...
Torin,-0.444409,0.172015,-1.959835,-3.965046,-1.919797,1.367929
Trametinib,1.094159,-0.737708,0.306731,0.325246,0.460066,-1.241208
U73122,-0.098850,-0.992238,0.136623,0.162417,0.213236,-1.690166


In [28]:
# Create dictionaries for x and y

# Store features and targets separately
X_data = {}
y_data = {}


In [29]:
# Split every kinase matrix

for kinase_id, matrix in feature_matrices.items():

    # Remove rows where target is missing
    matrix = matrix.dropna(subset=["KSEA_z_score"])

    # Features (all phosphosite FC values)
    X = matrix.drop(columns=["KSEA_z_score"])

    # Target (kinase activity)
    y = matrix["KSEA_z_score"]

    # Store
    X_data[kinase_id] = X
    y_data[kinase_id] = y

print(f"Prepared X and y for {len(X_data)} kinases.")

Prepared X and y for 85 kinases.


In [30]:
# Inspect one kinase

first_kinase = list(X_data.keys())[0]

print("Kinase:", first_kinase)

print("\nX shape:", X_data[first_kinase].shape)
print("y shape:", y_data[first_kinase].shape)

X_data[first_kinase].head()

Kinase: O00418

X shape: (61, 5)
y shape: (61,)


,EEF2K(S18),EEF2K(S66),EEF2K(S72),EEF2K(S74),EEF2K(Y69)
Perturbation,,,,,
AC220,0.664402,-1.563777,-1.686784,-1.405499,-0.623428
AT13148,0.045253,-0.191064,-0.219305,-0.263898,-0.317240
AZ20,0.250962,0.096789,-8.797069,-4.829727,-7.902993
AZD1480,-0.319168,-1.563777,-0.024642,-0.047332,-0.123976
AZD3759,-0.618604,-2.529060,-0.435913,-0.603194,-1.625165


In [31]:
# count missing values

missing_summary = {}

for kinase_id, X in X_data.items():
    
    missing_summary[kinase_id] = X.isna().sum().sum()

missing_df = (
    pd.DataFrame.from_dict(
        missing_summary,
        orient="index",
        columns=["Missing_FC_Values"] 
    )
    .reset_index()

)

missing_df.columns = ["UniprotID","Missing_FC_Values"]

missing_df.sort_values(
    by="Missing_FC_Values",
    ascending=False,
    inplace=True 
)

missing_df.head(10)

,UniprotID,Missing_FC_Values
0,O00418,0
54,Q15139,0
62,Q8IVT5,0
61,Q8IV63,0
60,Q7KZI7,0
59,Q2M2I8,0
58,Q16584,0
57,Q16513,0
56,Q16512,0
55,Q15418,0


In [32]:
missing_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 85 entries, 0 to 84
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   UniprotID          85 non-null     object
 1   Missing_FC_Values  85 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 2.0+ KB


In [33]:
print(missing_df.shape)

(85, 2)


In [34]:
#Feature scaling

X_scaled ={}

for kinase_id in X_data:

    scaler = StandardScaler()

    X_scaled[kinase_id] = pd.DataFrame(
        scaler.fit_transform(X_data[kinase_id]),
        columns=X_data[kinase_id].columns,
        index=X_data[kinase_id].index
    )

print(f"Scaled {len(X_scaled)} kinase feature matrices.")

Scaled 85 kinase feature matrices.


In [35]:
#inspect one kinase

first_kinase = list(X_scaled.keys())[0]

print(first_kinase)

X_scaled[first_kinase].head()

O00418


,EEF2K(S18),EEF2K(S66),EEF2K(S72),EEF2K(S74),EEF2K(Y69)
Perturbation,,,,,
AC220,1.442152,-0.700587,-0.699345,-0.584222,0.042448
AT13148,0.240362,0.518943,0.303391,0.316098,0.244829
AZ20,0.639650,0.774673,-5.557843,-3.284731,-4.769112
AZD1480,-0.466993,-0.700587,0.436405,0.486893,0.372570
AZD3759,-1.048209,-1.558151,0.155381,0.048514,-0.619668


In [36]:
#Train/test split

X_train = {}
X_test = {}
y_train = {}
y_test = {}

for kinase in X_scaled:

    X_train[kinase], X_test[kinase], y_train[kinase], y_test[kinase] = train_test_split(
        X_scaled[kinase],
        y_data[kinase],
        test_size=0.3,
        random_state=42
    )

print(f"Prepared train/test sets for {len(X_train)} kinases.")

Prepared train/test sets for 85 kinases.


In [37]:
#verify one kinase

first_kinase = list(X_train.keys())[0]

print("Kinase:", first_kinase)

print("X_train:",X_train[first_kinase].shape)
print("X_test:", X_test[first_kinase].shape)

print("y_train:", y_train[first_kinase].shape)
print("y_test:", y_test[first_kinase].shape)

Kinase: O00418
X_train: (42, 5)
X_test: (19, 5)
y_train: (42,)
y_test: (19,)


In [38]:
def evaluate_model(model, X_train, X_test, y_train, y_test):

    # Cross-validation
    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="r2"
    )

    # Train final model
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Metrics
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)

    return {
        "cv_mean": cv_scores.mean(),
        "cv_std": cv_scores.std(),
        "test_r2": r2,
        "test_mse": mse,
        "prediction": y_pred
    }

In [ ]:
rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

results = evaluate_model(
    rf,
    X_train[first_kinase],
    X_test[first_kinase],
    y_train[first_kinase],
    y_test[first_kinase]
)

print(results)

{'cv_mean': -0.13004585377930641, 'cv_std': 0.3081769931597794, 'test_r2': -0.6347705705542022, 'test_mse': 1.184970690594901, 'prediction': array([-1.05308627, -1.60366286, -1.82144145, -1.62615662, -1.39722863,
       -1.62088242, -1.7193239 , -1.62076751, -1.89717502,  0.25630276,
       -1.22948143, -2.00770013, -1.73098912, -1.85125249, -1.10118281,
       -1.21199037, -1.9384376 , -1.55418014, -1.56694064])}


In [39]:
def get_model(model_name, n_features=None):

    if model_name == "RandomForest":
        return RandomForestRegressor(
            n_estimators=100,
            random_state=42
        )

    elif model_name == "XGBoost":
        return XGBRegressor(
            objective="reg:squarederror",
            n_estimators=100,
            learning_rate=0.1,
            random_state=42
        )

    elif model_name == "PLS":

        n_components = min(3, n_features)

        return PLSRegression(
            n_components=n_components
        )

    elif model_name == "SVR":

        return SVR(
            kernel="linear",
            C=1.0,
            epsilon=0.1
        )

    elif model_name == "Lasso":

        return Lasso(
            alpha=0.1
        )

    elif model_name == "ElasticNet":

        return ElasticNet(
            alpha=0.1,
            l1_ratio=0.5
        )

    elif model_name == "GradientBoosting":

        return GradientBoostingRegressor(
            n_estimators=100,
            learning_rate=0.1,
            random_state=42
        )

    elif model_name == "MLP":

        hidden_size = min(max(n_features - 1, 1), 10)

        return MLPRegressor(
            hidden_layer_sizes=(hidden_size,),
            activation="relu",
            solver="adam",
            max_iter=5000,
            random_state=1
        )

    else:

        raise ValueError(f"Unknown model: {model_name}")

In [40]:
#Test the function

first_kinase = list(X_train.keys())[0]

print(first_kinase)

n_features = X_train[first_kinase].shape[1]

print(n_features)

O00418
5


In [41]:
#Build one random forest model
rf = get_model(
    "RandomForest",
    n_features
)

print(rf)

RandomForestRegressor(random_state=42)


In [42]:
#Build one MLP model
mlp = get_model(
    "MLP",
    n_features
)

print(mlp)

MLPRegressor(hidden_layer_sizes=(4,), max_iter=5000, random_state=1)


In [43]:
#Build one XGBoost model
xgb = get_model(
    "XGBoost",
    n_features
)

print(xgb)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)


In [44]:
#Build one PLS model
pls = get_model(
    "PLS",
    n_features
)

print(pls)

PLSRegression(n_components=3)


In [45]:
#Build one SVR model 
svr = get_model(
    "SVR",
    n_features
)

print(svr)

SVR(kernel='linear')


In [46]:
#Build one Gradient Boosting model
gbr = get_model(
    "GradientBoosting",
    n_features
)

print(gbr)

GradientBoostingRegressor(random_state=42)


In [47]:
#Build one Lasso model 
lasso = get_model(
    "Lasso",
    n_features
)

print(lasso)

Lasso(alpha=0.1)


In [48]:
#Build one ElasticNet model
enet = get_model(
    "ElasticNet",
    n_features
)

print(enet)

ElasticNet(alpha=0.1)


In [49]:
#Evaluate one model on first kinase
# First kinase
first_kinase = list(X_train.keys())[0]

# Number of phosphosite features
n_features = X_train[first_kinase].shape[1]

# Choose model
model = get_model(
    "RandomForest",
    n_features
)

# Evaluate model
results = evaluate_model(
    model,
    X_train[first_kinase],
    X_test[first_kinase],
    y_train[first_kinase],
    y_test[first_kinase]
)

results


{'cv_mean': -0.13004585377930641,
 'cv_std': 0.3081769931597794,
 'test_r2': -0.6347705705542022,
 'test_mse': 1.184970690594901,
 'prediction': array([-1.05308627, -1.60366286, -1.82144145, -1.62615662, -1.39722863,
        -1.62088242, -1.7193239 , -1.62076751, -1.89717502,  0.25630276,
        -1.22948143, -2.00770013, -1.73098912, -1.85125249, -1.10118281,
        -1.21199037, -1.9384376 , -1.55418014, -1.56694064])}

In [50]:
results_df = pd.DataFrame({
    "Metric": [
        "CV Mean R²",
        "CV Std R²",
        "Test R²",
        "Test MSE"
    ],
    "Value": [
        results["cv_mean"],
        results["cv_std"],
        results["test_r2"],
        results["test_mse"]
    ]
})

results_df

,Metric,Value
0,CV Mean R²,-0.130046
1,CV Std R²,0.308177
2,Test R²,-0.634771
3,Test MSE,1.184971


In [51]:
#comparing all 8 models automatically on the first kinase
#create the list of models 
model_names = [
    "RandomForest",
    "XGBoost",
    "PLS",
    "SVR",
    "Lasso",
    "ElasticNet",
    "GradientBoosting",
    "MLP"
]

In [52]:
#Create an empty list

comparison_results =[]

In [53]:
#Loop through every model
for model_name in model_names:

    model = get_model(
        model_name,
        n_features
    )

    results = evaluate_model(
        model,
        X_train[first_kinase],
        X_test[first_kinase],
        y_train[first_kinase],
        y_test[first_kinase]
    )

    comparison_results.append({
        "Model": model_name,
        "CV Mean R²": results["cv_mean"],
        "CV Std R²": results["cv_std"],
        "Test R²": results["test_r2"],
        "Test MSE": results["test_mse"]
    })

In [54]:
#Convert to a DataFrame

comparison_df = pd.DataFrame(comparison_results)

comparison_df

,Model,CV Mean R²,CV Std R²,Test R²,Test MSE
0,RandomForest,-0.130046,0.308177,-0.634771,1.184971
1,XGBoost,-0.543112,0.278270,-1.498010,1.810693
2,PLS,0.045130,0.427042,-0.401425,1.015829
3,SVR,0.028713,0.430091,-0.542878,1.118362
4,Lasso,0.005693,0.265054,-0.126371,0.816455
5,ElasticNet,0.015078,0.277557,-0.194098,0.865547
6,GradientBoosting,-0.194652,0.366663,-0.981469,1.436277
7,MLP,0.173554,0.319733,-0.616617,1.171812


In [55]:
#Sort from best to worst
comparison_df = comparison_df.sort_values(
    by="Test R²",
    ascending=False
)

comparison_df

,Model,CV Mean R²,CV Std R²,Test R²,Test MSE
4,Lasso,0.005693,0.265054,-0.126371,0.816455
5,ElasticNet,0.015078,0.277557,-0.194098,0.865547
2,PLS,0.045130,0.427042,-0.401425,1.015829
3,SVR,0.028713,0.430091,-0.542878,1.118362
7,MLP,0.173554,0.319733,-0.616617,1.171812
0,RandomForest,-0.130046,0.308177,-0.634771,1.184971
6,GradientBoosting,-0.194652,0.366663,-0.981469,1.436277
1,XGBoost,-0.543112,0.278270,-1.498010,1.810693


In [56]:
#Create an empty results Dataframe

all_results = pd.DataFrame(columns=[
    "Kinase",
    "Model",
    "CV Mean R²",
    "CV Std R²",
    "Test R²",
    "Test MSE"
])

In [57]:
#Create the list of model names

model_names = [
    "RandomForest",
    "XGBoost",
    "PLS",
    "SVR",
    "Lasso",
    "ElasticNet",
    "GradientBoosting",
    "MLP"
]

In [58]:
gene_lookup = (
    usable_df[["UniprotID", "GeneName"]]
    .drop_duplicates()
    .set_index("UniprotID")["GeneName"]
    .to_dict()
)

In [59]:
prediction_results = []

In [61]:
#Loop through all 85 kinases and all 8 models

for kinase in X_train.keys():

    print(f"\nProcessing kinase: {kinase}")

    # Number of phosphosite features for this kinase
    n_features = X_train[kinase].shape[1]

    # Loop through all models
    for model_name in model_names:

        print(f"   Running {model_name}...")

        # Create a fresh model
        model = get_model(model_name, n_features)

        # Evaluate the model
        results = evaluate_model(
            model,
            X_train[kinase],
            X_test[kinase],
            y_train[kinase],
            y_test[kinase]
        )

        for actual, predicted in zip(y_test[kinase],results["prediction"]):
            
            prediction_results.append({
                "Kinase": kinase,
                "GeneName": gene_lookup.get(kinase, "Unkown"),
                "Model": model_name,
                "Actual": actual,
                "Predicted": predicted 
            })

        # Save results
        all_results.loc[len(all_results)] = [
            kinase,
            model_name,
            results["cv_mean"],
            results["cv_std"],
            results["test_r2"],
            results["test_mse"]
        ]


Processing kinase: O00418
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running GradientBoosting...
   Running MLP...

Processing kinase: O14578
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running GradientBoosting...
   Running MLP...

Processing kinase: O15075
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running GradientBoosting...
   Running MLP...

Processing kinase: O43318
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running GradientBoosting...
   Running MLP...

Processing kinase: O43353
   Running RandomForest...
   Running XGBoost...
   Running PLS...
   Running SVR...
   Running Lasso...
   Running ElasticNet...
   Running Gradient

In [62]:
prediction_df = pd.DataFrame(prediction_results)

prediction_df.to_csv(
    "all_predictions.csv",
    index=False
)

print(prediction_df.shape)

prediction_df.head()

(12920, 5)


,Kinase,GeneName,Model,Actual,Predicted
0,O00418,EEF2K,RandomForest,-1.904592,-1.053086
1,O00418,EEF2K,RandomForest,-1.109755,-1.603663
2,O00418,EEF2K,RandomForest,-1.312516,-1.821441
3,O00418,EEF2K,RandomForest,-0.818783,-1.626157
4,O00418,EEF2K,RandomForest,-1.587355,-1.397229


In [63]:
#check the size of the results

all_results.shape

(680, 6)

In [64]:
all_results

,Kinase,Model,CV Mean R²,CV Std R²,Test R²,Test MSE
0,O00418,RandomForest,-0.130046,0.308177,-0.634771,1.184971
1,O00418,XGBoost,-0.543112,0.278270,-1.498010,1.810693
2,O00418,PLS,0.045130,0.427042,-0.401425,1.015829
3,O00418,SVR,0.028713,0.430091,-0.542878,1.118362
4,O00418,Lasso,0.005693,0.265054,-0.126371,0.816455
...,...,...,...,...,...,...
675,Q9Y3S1,SVR,-0.142517,0.086057,-0.152896,3.507064
676,Q9Y3S1,Lasso,-0.114919,0.052052,-0.012598,3.080283
677,Q9Y3S1,ElasticNet,-0.127093,0.067681,-0.032379,3.140455
678,Q9Y3S1,GradientBoosting,0.263020,0.462172,-0.455381,4.427209


In [65]:
all_results.to_csv(
    "all_models_85_kinases.csv",
    index=False
)

print("Saved!")

Saved!


In [66]:
#which model wins for each kinase

best_models = (
    all_results
    .sort_values("Test R²", ascending=False)
    .groupby("Kinase")
    .first()
    .reset_index()
)

best_models

,Kinase,Model,CV Mean R²,CV Std R²,Test R²,Test MSE
0,O00418,Lasso,0.005693,0.265054,-0.126371,0.816455
1,O14578,RandomForest,-0.611219,0.755537,0.023617,0.480279
2,O15075,Lasso,0.200674,0.369384,0.122552,0.938032
3,O43318,GradientBoosting,-1.415528,2.148830,0.230035,0.643804
4,O43353,PLS,0.667993,0.148958,0.556355,0.179804
...,...,...,...,...,...,...
80,Q9NYV4,Lasso,-0.141184,0.589356,-0.355345,0.387042
81,Q9UKE5,PLS,0.965607,0.015750,0.974440,0.019708
82,Q9Y2K2,RandomForest,-0.145324,0.216990,0.104651,0.955756
83,Q9Y2U5,GradientBoosting,-1.013220,0.823379,0.047494,3.607197


In [67]:
#count how many kinases each model wins

best_models["Model"].value_counts()

Model
Lasso               21
PLS                 15
SVR                 15
RandomForest        13
MLP                 10
GradientBoosting     6
ElasticNet           5
Name: count, dtype: int64

In [68]:
best_models.to_csv(
    "best_model_per_kinase.csv",
    index=False
)